[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/mihiarc/socialmapper/blob/main/docs/notebooks/01-isochrones-getting-started.ipynb)

# Getting Started with Isochrones

## What is an isochrone?

An **isochrone** (from Greek *iso* = equal, *chronos* = time) is a polygon on a map showing all the places reachable from a starting point within a given amount of time. Think of it as answering the question:

> *"If I leave from this spot right now, how far can I get in 15 minutes?"*

Unlike a simple radius circle, an isochrone follows the actual road network — so the shape is irregular, stretching further along highways and compressing near dead-ends or winding streets.

### Why isochrones matter

Isochrones are a foundational tool in **spatial equity analysis**, **urban planning**, and **site selection**:

- **Healthcare access** — What percentage of a city can reach an emergency room within 10 minutes?
- **Food deserts** — Can residents walk to a grocery store in 15 minutes?
- **Transit planning** — How does a new bus route change the area reachable by public transit?
- **Real estate** — What amenities are within a 20-minute drive of a property?

### How SocialMapper generates isochrones

SocialMapper uses the [Valhalla](https://valhalla.github.io/valhalla/) open-source routing engine. Valhalla is the same engine used by Mapbox and many city governments. Key benefits:

- **Free** — no API key, no usage limits
- **Three travel modes** — `'drive'`, `'walk'`, `'bike'`
- **Realistic routing** — follows actual roads, respects one-way streets, turn restrictions, and speed limits

### What you will learn

1. Create your first isochrone from a city name
2. Understand the GeoJSON result format
3. Use coordinate tuples as input
4. Compare travel modes (drive / walk / bike)
5. Compare travel times (5, 10, 15, 30 minutes)
6. Visualize isochrones with matplotlib

## Setup

In [ ]:
# Uncomment to install on Google Colab:
# !pip install 'socialmapper @ git+https://github.com/mihiarc/socialmapper.git'

from socialmapper import create_isochrone
import matplotlib.pyplot as plt
from shapely.geometry import shape
import geopandas as gpd
import contextily as cx

---
## 1. Your First Isochrone

The simplest way to create an isochrone is to pass a `"City, State"` string. SocialMapper will geocode it (convert the name to latitude/longitude coordinates) automatically using the Nominatim geocoding service.

The three parameters are:
- **`location`** — where to start (city name or coordinates)
- **`travel_time`** — how many minutes of travel (1–120)
- **`travel_mode`** — how you're traveling (`'drive'`, `'walk'`, or `'bike'`)

In [ ]:
iso = create_isochrone("Nashville, TN", travel_time=15, travel_mode="drive")
print(f"Type: {type(iso).__name__}")
print(f"Keys: {list(iso.keys())}")

The result is a plain Python dictionary in **GeoJSON Feature** format — a standard geospatial data format used by mapping libraries worldwide.

---
## 2. Understanding the GeoJSON Result

Every GeoJSON Feature has three parts:

| Key | What it contains |
|---|---|
| `type` | Always `"Feature"` — identifies this as a GeoJSON Feature |
| `geometry` | The polygon shape (list of coordinate pairs forming the boundary) |
| `properties` | Metadata — location name, travel time, mode, area, etc. |

Let's inspect each part.

In [ ]:
# Properties tell you about the isochrone
print("=== Properties ===")
for key, value in iso["properties"].items():
    print(f"  {key}: {value}")

The `area_sq_km` property tells you how much land area is reachable — this is one of the most useful metrics for comparing accessibility across locations.

In [ ]:
# The geometry is a GeoJSON Polygon — a list of coordinate rings
geom = iso["geometry"]
print(f"Geometry type: {geom['type']}")
print(f"Coordinate rings: {len(geom['coordinates'])}")
print(f"Vertices in boundary: {len(geom['coordinates'][0])}")
print(f"First vertex (lon, lat): {geom['coordinates'][0][0]}")

> **Note:** GeoJSON uses `[longitude, latitude]` order (not lat, lon). This is a common source of confusion — just remember that GeoJSON follows the mathematical convention of `[x, y]`.

Let's do a quick visualization of the polygon to see what 15 minutes of driving from Nashville looks like.

In [ ]:
polygon = shape(iso["geometry"])

fig, ax = plt.subplots(figsize=(8, 8))
gdf = gpd.GeoDataFrame(geometry=[polygon], crs="EPSG:4326")
gdf.plot(ax=ax, alpha=0.3, color="steelblue", edgecolor="steelblue", linewidth=1)
cx.add_basemap(ax, crs="EPSG:4326", source=cx.providers.CartoDB.Positron)

# Mark the center point
centroid = polygon.centroid
ax.plot(centroid.x, centroid.y, 'r*', markersize=15, label="Origin", zorder=5)

ax.set_title(f"15-Minute Drive from Nashville, TN\n({iso['properties']['area_sq_km']:.1f} sq km)", fontsize=13)
ax.set_xlabel("Longitude")
ax.set_ylabel("Latitude")
ax.legend()
plt.tight_layout()
plt.show()

Notice how the shape is **not a circle** — it stretches along major highways (like I-40 and I-24 in Nashville) and is smaller in areas with slower local streets. This is what makes isochrones more realistic than simple radius buffers.

---
## 3. Coordinate Tuple Input

Instead of a city name, you can pass exact `(latitude, longitude)` coordinates. This is useful when you want to analyze a specific address, intersection, or landmark rather than a city center.

> **Important:** The tuple order is `(latitude, longitude)` — the opposite of GeoJSON's internal `[lon, lat]` format. SocialMapper uses `(lat, lon)` because it's more intuitive and matches Google Maps, GPS devices, and most mapping APIs.

In [ ]:
# Centennial Park, Nashville
iso_coords = create_isochrone((36.1496, -86.8130), travel_time=10, travel_mode="drive")
print(f"Location: {iso_coords['properties']['location']}")
print(f"Area: {iso_coords['properties']['area_sq_km']:.1f} sq km")

---
## 4. Compare Travel Modes

The travel mode dramatically affects how far you can go. The three modes use different parts of the road network and different speeds:

| Mode | Uses | Typical speed |
|---|---|---|
| `'drive'` | Roads, highways | 25–65 mph depending on road type |
| `'bike'` | Roads, bike lanes, paths | 10–15 mph |
| `'walk'` | Sidewalks, paths, crosswalks | 3–4 mph |

Let's compare all three modes from the same origin with the same 15-minute time budget.

In [ ]:
modes = ["drive", "walk", "bike"]
mode_results = {}

for mode in modes:
    result = create_isochrone("Nashville, TN", travel_time=15, travel_mode=mode)
    area = result["properties"]["area_sq_km"]
    mode_results[mode] = result
    print(f"{mode:>5}: {area:>8.1f} sq km")

In [ ]:
# Side-by-side polygon comparison with road basemaps
fig, axes = plt.subplots(1, 3, figsize=(18, 6))
colors = {"drive": "#2171b5", "walk": "#238b45", "bike": "#d94801"}

for ax, mode in zip(axes, modes):
    polygon = shape(mode_results[mode]["geometry"])
    gdf = gpd.GeoDataFrame(geometry=[polygon], crs="EPSG:4326")
    gdf.plot(ax=ax, alpha=0.3, color=colors[mode], edgecolor=colors[mode], linewidth=1)
    cx.add_basemap(ax, crs="EPSG:4326", source=cx.providers.CartoDB.Positron)
    centroid = polygon.centroid
    ax.plot(centroid.x, centroid.y, 'k*', markersize=12, zorder=5)
    area = mode_results[mode]["properties"]["area_sq_km"]
    ax.set_title(f"{mode.title()} — {area:.1f} sq km", fontsize=12)
    ax.tick_params(labelsize=7)

fig.suptitle("15-Minute Isochrones from Nashville, TN", fontsize=14, fontweight="bold")
plt.tight_layout()
plt.show()

The difference is striking — in 15 minutes of driving you can cover a huge area, while walking limits you to a small neighborhood. This disparity is at the heart of **transportation equity**: people without cars have dramatically less access to jobs, services, and amenities.

In [ ]:
# Bar chart comparing reachable area by mode
areas = [mode_results[m]["properties"]["area_sq_km"] for m in modes]

fig, ax = plt.subplots(figsize=(7, 4))
bars = ax.bar(modes, areas, color=[colors[m] for m in modes], edgecolor="white", linewidth=1.5)

# Add value labels on bars
for bar, area in zip(bars, areas):
    ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 1,
            f"{area:.1f}", ha="center", va="bottom", fontweight="bold")

ax.set_ylabel("Reachable Area (sq km)")
ax.set_title("15-Minute Reachable Area by Travel Mode", fontsize=13)
ax.set_ylim(0, max(areas) * 1.15)
plt.tight_layout()
plt.show()

---
## 5. Compare Travel Times

How does the reachable area grow as we increase the travel time? The relationship is not linear — doubling the time more than doubles the area because you can reach further out along highways, opening up much more land.

Let's generate driving isochrones at 5, 10, 15, and 30 minutes.

In [ ]:
travel_times = [5, 10, 15, 30]
time_results = {}

for minutes in travel_times:
    result = create_isochrone("Nashville, TN", travel_time=minutes, travel_mode="drive")
    area = result["properties"]["area_sq_km"]
    time_results[minutes] = result
    print(f"{minutes:>2} min drive: {area:>8.1f} sq km")

In [ ]:
# Nested isochrone plot with road basemap — largest first so smaller ones draw on top
fig, ax = plt.subplots(figsize=(9, 9))
cmap = plt.cm.Reds
color_values = [0.2, 0.4, 0.6, 0.85]

for minutes, cval in zip(reversed(travel_times), reversed(color_values)):
    polygon = shape(time_results[minutes]["geometry"])
    gdf = gpd.GeoDataFrame(geometry=[polygon], crs="EPSG:4326")
    color = cmap(cval)
    area = time_results[minutes]["properties"]["area_sq_km"]
    gdf.plot(ax=ax, alpha=0.5, color=color, edgecolor=color, linewidth=0.8,
             label=f"{minutes} min ({area:.0f} sq km)")

cx.add_basemap(ax, crs="EPSG:4326", source=cx.providers.CartoDB.Positron)

ax.set_title("Driving Isochrones from Nashville, TN", fontsize=14, fontweight="bold")
ax.legend(loc="upper right", fontsize=10)
ax.set_xlabel("Longitude")
ax.set_ylabel("Latitude")
plt.tight_layout()
plt.show()

In [ ]:
# Area growth curve — shows the non-linear relationship
time_list = sorted(time_results.keys())
area_list = [time_results[t]["properties"]["area_sq_km"] for t in time_list]

fig, ax = plt.subplots(figsize=(7, 4))
ax.plot(time_list, area_list, 'o-', color="#cb181d", linewidth=2, markersize=8)

for t, a in zip(time_list, area_list):
    ax.annotate(f"{a:.0f} km\u00b2", (t, a), textcoords="offset points",
                xytext=(10, 5), fontsize=9)

ax.set_xlabel("Travel Time (minutes)")
ax.set_ylabel("Reachable Area (sq km)")
ax.set_title("How Reachable Area Grows with Travel Time (Driving)", fontsize=13)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

The growth is roughly **quadratic** — doubling the travel time roughly quadruples the reachable area. This makes intuitive sense: you can go twice as far in each direction, covering an area proportional to the square of the distance.

This has important implications for policy: a 5-minute improvement in commute time (from a new road or transit line) yields a much larger increase in accessible area than you might expect.

---
## 6. All Three Modes Overlaid

Finally, let's overlay all three travel modes on the same map to see the accessibility gap clearly.

In [ ]:
fig, ax = plt.subplots(figsize=(9, 9))

# Plot in order: drive (largest) then bike then walk (smallest on top)
for mode, color, alpha in [("drive", "#2171b5", 0.2), ("bike", "#d94801", 0.3), ("walk", "#238b45", 0.4)]:
    polygon = shape(mode_results[mode]["geometry"])
    gdf = gpd.GeoDataFrame(geometry=[polygon], crs="EPSG:4326")
    area = mode_results[mode]["properties"]["area_sq_km"]
    gdf.plot(ax=ax, alpha=alpha, color=color, edgecolor=color, linewidth=1,
             label=f"{mode} ({area:.1f} sq km)")

cx.add_basemap(ax, crs="EPSG:4326", source=cx.providers.CartoDB.Positron)

ax.set_title("15-Minute Reachable Area — All Modes\nNashville, TN", fontsize=14, fontweight="bold")
ax.legend(loc="upper right", fontsize=11)
plt.tight_layout()
plt.show()

---
## Summary

### Key concepts
- An **isochrone** shows the area reachable within a travel time — more realistic than a simple circle
- Reachable area grows **quadratically** with travel time
- **Travel mode** dramatically affects accessibility — the gap between driving and walking is an equity issue

### API reference

| What you learned | API |
|---|---|
| Create an isochrone from a city name | `create_isochrone("Nashville, TN", travel_time=15)` |
| Create an isochrone from coordinates | `create_isochrone((36.16, -86.78), travel_time=15)` |
| Change travel mode | `travel_mode='drive'` \| `'walk'` \| `'bike'` |
| Read the result | `iso['properties']` for metadata, `iso['geometry']` for the shape |

**Next notebook:** [02 — Census Block Groups](02-census-block-groups.ipynb) — learn how to find the census geographies inside an isochrone